# 🛒 Рекомендательная система товаров в электронной коммерции

## 📌 Описание проекта

Электронная коммерция является одной из ключевых областей применения рекомендательных систем. Такие системы позволяют пользователям быстрее находить интересующие их товары, а бизнесу — увеличивать вовлечённость и выручку.

В рамках данного проекта решается задача:

> **предсказания товаров, которые с наибольшей вероятностью будут добавлены пользователем в корзину**

Рекомендации строятся на основе пользовательских взаимодействий:

* просмотры (`view`)
* добавления в корзину (`addtocart`)
* покупки (`transaction`)

---

## 🎯 Цель проекта

Разработать рекомендательную систему, которая:

* предсказывает товары, интересные пользователю
* учитывает историю взаимодействий
* работает в условиях разреженных данных
* обрабатывает cold start
* может быть развернута как веб-сервис

---

## 🧠 Постановка задачи

Задача формулируется как:

> **ranking / recommendation задача с implicit feedback**

Особенности:

* неявная обратная связь (implicit)
* дисбаланс событий (`view >> addtocart >> transaction`)
* временная структура данных
* высокая разреженность user-item матрицы

---

## 📊 Данные

Датасет состоит из трёх источников:

### 1. `events.csv`

Лог пользовательских событий:

* `timestamp` — время события
* `visitorid` — пользователь
* `event` — тип события (`view`, `addtocart`, `transaction`)
* `itemid` — товар
* `transactionid` — id покупки

---

### 2. `category_tree.csv`

* `categoryid` — категория
* `parentid` — родительская категория

---

### 3. `item_properties`

Файл разбит на две части:

* `item_properties_part1.csv`
* `item_properties_part2.csv`

Содержит:

* `timestamp`
* `itemid`
* `property`
* `value`

---

📎 **Ссылка на датасет:**
*(добавь сюда ссылку)*

---

## 🏗️ Структура проекта

```bash
ecommerce-recsys/
├── airflow/
│   └── dags/
├── mlflow/
│   ├── mlruns/
│   └── mlflow.db
├── artifacts/
│   └── models/
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
│   ├── 01_eda.ipynb
│   └── 02_modeling.ipynb
├── src/
│   ├── api/
│   ├── data/
│   ├── models/
│   ├── monitoring/
│   └── utils/
├── scripts/
├── docker/
├── README.md
└── requirements.txt
```

---

## 🔍 Этапы проекта

### 1. EDA

* анализ структуры данных
* распределение событий
* анализ активности пользователей
* популярность товаров
* анализ разреженности
* cold start

📄 `notebooks/01_eda.ipynb`

---

### 2. Моделирование

* baseline: Top Popular
* ALS (collaborative filtering)
* генерация кандидатов
* оценка метрик

📄 `notebooks/02_modeling.ipynb`

---

### 3. Инфраструктура

* MLflow для экспериментов
* Airflow для переобучения
* Docker для деплоя

---

### 4. API

* сервис рекомендаций на FastAPI
* endpoint `/recommend`

---

### 5. Мониторинг

* технические метрики
* метрики качества рекомендаций

---

## 📏 Метрики

Основные:

* Recall@K
* MAP@K
* NDCG@K

---

## ⚙️ Используемые технологии

* Python (pandas, numpy)
* implicit (ALS)
* scikit-learn
* LightGBM (расширение)
* MLflow
* FastAPI
* Docker
* Apache Airflow

---

## ⚠️ Особенности задачи

* сильный дисбаланс событий
* long-tail распределение товаров
* высокая разреженность данных
* cold start пользователей
* необходимость гибридного подхода

---

## 🚀 Запуск проекта

### 1. Установка

```bash
git clone <repo>
cd ecommerce-recsys

python -m venv .venv
source .venv/bin/activate

pip install -r requirements.txt
```

---

### 2. Подготовка данных

Положить данные в:

```bash
data/raw/
```

---

### 3. Запуск EDA

```bash
jupyter notebook
```

---

### 4. MLflow

```bash
bash scripts/run_mlflow.sh
```

---

### 5. API

```bash
uvicorn src.api.app:app --reload
```

---

## 📦 Результат

В результате проекта реализована система, которая:

* учитывает поведение пользователей
* предсказывает интересующие товары
* комбинирует несколько подходов (ALS + популярность)
* готова к деплою и масштабированию


# 1. Импорты

In [1]:
# Стандартная библиотека
import gc
import logging
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple
import os
import sys
import warnings

import pandas as pd

# Работа с данными
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Статистика
from scipy.sparse import csr_matrix
from scipy.stats import gaussian_kde

# ML
import lightgbm as lgb
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Файлы и хранилища
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from dotenv import load_dotenv

# Прочее
import phik
from phik import report, resources
from tqdm.auto import tqdm

In [2]:
# настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.0f}".format

# настройки графиков
%matplotlib inline
%config InlineBackend.figure_format = "retina"

# корень проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", os.path.basename(PROJECT_ROOT))
print("src exists:", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))

def to_relative(path, base):
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path

from src.utils.config import (
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    EVENTS_PATH,
    CATEGORY_TREE_PATH,
    ITEM_PROPERTIES_PATH,
    ARTIFACTS_DIR,
    MODELS_DIR,
    MLFLOW_BASE_DIR,
    MLFLOW_DIR,
    MLFLOW_DB_PATH,
    AIRFLOW_DIR,
    AIRFLOW_DAGS_DIR,
)

# S3
S3_BASE = "s3://s3-student-mle-20250927-31ecef0a74/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

# проверка путей
paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "MLFLOW_BASE_DIR": MLFLOW_BASE_DIR,
    "MLFLOW_DIR": MLFLOW_DIR,
    "MLFLOW_DB_PATH": MLFLOW_DB_PATH,
    "AIRFLOW_DIR": AIRFLOW_DIR,
    "AIRFLOW_DAGS_DIR": AIRFLOW_DAGS_DIR,
    "EVENTS_PATH": EVENTS_PATH,
    "CATEGORY_TREE_PATH": CATEGORY_TREE_PATH,
    "ITEM_PROPERTIES_PATH": ITEM_PROPERTIES_PATH,
}

print("\nProject paths:\n")

for name, path in paths_to_check.items():
    rel_path = to_relative(path, PROJECT_ROOT)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"{name:<22} {rel_path:<40} [{status}]")

PROJECT_ROOT: ecommerce-recsys
src exists: True

Project paths:

DATA_DIR               data                                     [OK]
RAW_DIR                data/raw                                 [OK]
PROCESSED_DIR          data/processed                           [OK]
ARTIFACTS_DIR          artifacts                                [OK]
MODELS_DIR             artifacts/models                         [OK]
MLFLOW_BASE_DIR        mlflow                                   [OK]
MLFLOW_DIR             mlflow/mlruns                            [OK]
MLFLOW_DB_PATH         mlflow/mlflow.db                         [MISSING]
AIRFLOW_DIR            airflow                                  [OK]
AIRFLOW_DAGS_DIR       airflow/dags                             [OK]
EVENTS_PATH            data/raw/events.csv                      [OK]
CATEGORY_TREE_PATH     data/raw/category_tree.csv               [OK]
ITEM_PROPERTIES_PATH   data/raw/item_properties.csv             [MISSING]


# 2. Загрузка данных

In [ ]:
events = pd.read_csv(
    EVENTS_PATH,
    dtype={
        "visitorid": "int32",
        "event": "category",
        "itemid": "int32",
        "transactionid": "float64",
    },
)
events["timestamp"] = pd.to_datetime(events["timestamp"], unit="ms")

category_tree = pd.read_csv(
    f"{RAW_DIR}/category_tree.csv",
    dtype={
        "categoryid": "int32",
        "parentid": "float64",
    },
)

item_props_part1 = pd.read_csv(
    f"{RAW_DIR}/item_properties_part1.csv",
    dtype={
        "itemid": "int32",
        "property": "category",
        "value": "string",
    },
)

item_props_part2 = pd.read_csv(
    f"{RAW_DIR}/item_properties_part2.csv",
    dtype={
        "itemid": "int32",
        "property": "category",
        "value": "string",
    },
)

item_props = pd.concat([item_props_part1, item_props_part2], ignore_index=True)
item_props["timestamp"] = pd.to_datetime(item_props["timestamp"], unit="ms")

print(events.shape, category_tree.shape, item_props.shape)

(2756101, 5) (1669, 2) (20275902, 4)


# 3. Общая информация

In [ ]:
print("Events shape:", events.shape)
print("Category tree shape:", category_tree.shape)
print("Item properties shape:", item_props.shape)

events.head()

# 4. Типы данных и пропуски

In [ ]:
events.info()

events.isnull().sum()

Вывод (написать текстом):

есть ли пропуски

где они

насколько критично

# 5. Уникальные значения

In [ ]:
print("Users:", events["visitorid"].nunique())
print("Items:", events["itemid"].nunique())
print("Events:", events.shape[0])

# 6. Распределение типов событий

In [ ]:
events["event"].value_counts()

In [ ]:
sns.countplot(data=events, x="event")
plt.title("Event distribution")
plt.show()

In [ ]:
Вывод:
обычно view >> addtocart >> transaction
сильный дисбаланс

# 7. Временной анализ

In [ ]:
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")

events["date"] = events["datetime"].dt.date
events["hour"] = events["datetime"].dt.hour

## События по времени

In [ ]:
events.groupby("date").size().plot(figsize=(12,4), title="Events over time")
plt.show()

## По часам

In [ ]:
sns.countplot(data=events, x="hour")
plt.title("Events by hour")
plt.show()

Вывод:
пики активности
есть ли сезонность

# 8. Популярные товары

In [ ]:
top_items = events["itemid"].value_counts().head(20)

top_items.plot(kind="bar", figsize=(10,4), title="Top items")
plt.show()

Вывод:
есть сильный popularity bias

# 9. Активность пользователей

In [ ]:
user_activity = events.groupby("visitorid").size()

user_activity.describe()

In [ ]:
sns.histplot(user_activity, bins=50, log_scale=True)
plt.title("User activity distribution")
plt.show()

In [ ]:
Вывод:
большинство пользователей малоактивны
длинный хвост

# 10. Активность товаров

In [ ]:
item_activity = events.groupby("itemid").size()

sns.histplot(item_activity, bins=50, log_scale=True)
plt.title("Item popularity distribution")
plt.show()

In [ ]:
Вывод:
few popular items, many rare

# 11. Конверсия

In [ ]:
events["event"].value_counts(normalize=True)

In [ ]:
Вывод:
низкая конверсия в покупку
add-to-cart — разумный target

# 12. Разреженность (sparsity)

In [ ]:
n_users = events["visitorid"].nunique()
n_items = events["itemid"].nunique()
n_interactions = len(events)

sparsity = 1 - n_interactions / (n_users * n_items)

print("Sparsity:", sparsity)

In [ ]:
Вывод:
обычно ~99%+
значит нужен CF/ALS

# 13. Cold start

In [ ]:
user_counts = events["visitorid"].value_counts()
cold_users = (user_counts == 1).mean()

print("Cold users ratio:", cold_users)

In [ ]:
Вывод:
много новых пользователей
нужен popular fallback

# 14. Анализ item_properties

In [ ]:
item_props["property"].value_counts().head(10)

In [ ]:
item_props["value"].nunique()

In [ ]:
Вывод:
данные разреженные
можно использовать как фичи

# Финальные выводы

In [ ]:
1. Данные имеют сильный дисбаланс: большинство событий — просмотры.
2. Пользовательская активность распределена неравномерно (long tail).
3. Наблюдается сильный popularity bias среди товаров.
4. Матрица user-item сильно разрежена (>99%).
5. Высокая доля cold users → нужен fallback (Top Popular).
6. Добавление в корзину выбрано как целевое действие.
7. Подход к решению:
   - candidate generation (ALS, popularity)
   - ranking (LightGBM)